# 🎓 Classroom Abnormal Behavior Recognition
## Phase 1 (CNN) + Phase 2 (Vision Transformer)

| Feature | Status |
|---|---|
| MobileNetV2 Transfer Learning + Fine-Tuning | ✅ |
| Vision Transformer (ViT from scratch) | ✅ |
| Out-of-Distribution (OOD) Detection | ✅ |
| Unknown / No-Person image rejection | ✅ |
| Strong Augmentation | ✅ |
| Grad-CAM Explainability | ✅ |
| Full Metrics: Confusion Matrix, F1, Precision, Recall | ✅ |

### Problem being solved
The previous model predicted a class for **every** image including empty rooms,
random objects, and non-classroom images. This notebook adds:
1. **Confidence thresholding** — reject predictions below a calibrated threshold
2. **Entropy-based OOD detection** — flag images that look nothing like training data
3. **Vision Transformer** — second model for comparison and ensemble


## 📁 Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 2 — Install Extra Libraries

`vit-keras` provides a ready-made Vision Transformer compatible with TF/Keras.

In [ ]:
# vit-keras: Vision Transformer implementation for TensorFlow
!pip install -q vit-keras tensorflow-addons
print('Installation complete.')

## 📦 Step 3 — Imports

In [ ]:
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing import image as keras_image

from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, accuracy_score
)
from scipy.stats import entropy as scipy_entropy

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## ⚙️ Step 4 — Configuration

All hyper-parameters in one place. The ViT config is separate from CNN config.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR  = '/content/drive/MyDrive/DeepLearningProject/dataset'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'validation')
TEST_DIR  = os.path.join(BASE_DIR, 'test')
SAVE_DIR  = '/content/drive/MyDrive/DeepLearningProject/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Shared ────────────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)
IMG_SHAPE   = (224, 224, 3)
BATCH_SIZE  = 16
NUM_CLASSES = 5
SEED        = 42

# ── CNN (MobileNetV2) config ──────────────────────────────────────────────
CNN_EPOCHS_FROZEN   = 30
CNN_EPOCHS_FINETUNE = 20
CNN_LR_FROZEN       = 1e-3
CNN_LR_FINETUNE     = 1e-5
CNN_UNFREEZE_FROM   = -30
CNN_DROPOUT         = 0.4

# ── Vision Transformer config ─────────────────────────────────────────────
VIT_PATCH_SIZE      = 16       # each patch = 16x16 pixels
VIT_NUM_PATCHES     = (224 // 16) ** 2   # = 196 patches
VIT_PROJECTION_DIM  = 64       # embedding dimension per patch
VIT_NUM_HEADS       = 4        # multi-head attention heads
VIT_TRANSFORMER_LAYERS = 8     # number of transformer encoder blocks
VIT_MLP_HEAD_UNITS  = [2048, 1024]  # dense units in classification head
VIT_EPOCHS          = 50
VIT_LR              = 1e-4
VIT_DROPOUT         = 0.1

# ── OOD (Out-of-Distribution) detection config ────────────────────────────
CONFIDENCE_THRESHOLD = 0.60    # predictions below this → 'Unknown / No behaviour'
ENTROPY_THRESHOLD    = 1.40    # high entropy → image looks nothing like training data
# Max possible entropy for 5 classes = log(5) ≈ 1.609

print('Configuration ready.')

## 📂 Step 5 — Load Datasets

Rescaling is placed **inside the model**, so raw 0-255 pixels are passed here.
This guarantees identical preprocessing at training and inference time.

In [ ]:
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = train_ds_raw.class_names
print(f'Classes detected: {CLASS_NAMES}')

with open(os.path.join(SAVE_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f)
print('class_names.json saved.')

## 🔀 Step 6 — Augmentation Pipeline

Strong augmentation is essential with only 300 images per class.
Applied only to training data, never to val/test.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(0.15, seed=SEED),
    layers.RandomZoom(0.15, seed=SEED),
    layers.RandomTranslation(0.10, 0.10, seed=SEED),
    layers.RandomContrast(0.30, seed=SEED),
    layers.RandomBrightness(0.20, seed=SEED),
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE

def augment_train(image, label):
    return data_augmentation(image, training=True), label

train_ds = train_ds_raw.map(augment_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds  = test_ds_raw.cache().prefetch(AUTOTUNE)

print('tf.data pipelines ready.')

## 🏗️ Step 7 — Build CNN Model (MobileNetV2)

The `Rescaling(1/255)` layer lives **inside** the model.
This means you always pass raw 0–255 images — no manual `/255` needed anywhere,
in training, inference, or the web app backend.

```
Input (0-255) → Rescaling(1/255) → MobileNetV2 → GAP → BN → Dense(256) → Dropout → Dense(128) → Dropout → Softmax(5)
```

In [ ]:
def build_cnn_model():
    inputs  = keras.Input(shape=IMG_SHAPE, name='input_image')
    x       = layers.Rescaling(1.0/255.0, name='rescaling')(inputs)
    backbone = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights='imagenet')
    backbone.trainable = False
    x       = backbone(x, training=False)
    x       = layers.GlobalAveragePooling2D(name='gap')(x)
    x       = layers.BatchNormalization(name='head_bn')(x)
    x       = layers.Dense(256, activation='relu', name='dense_256')(x)
    x       = layers.Dropout(CNN_DROPOUT, name='dropout_1')(x)
    x       = layers.Dense(128, activation='relu', name='dense_128')(x)
    x       = layers.Dropout(0.3, name='dropout_2')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)
    return keras.Model(inputs, outputs, name='CNN_MobileNetV2'), backbone

cnn_model, cnn_backbone = build_cnn_model()
cnn_model.summary(line_length=90)

## 🚀 Step 8 — CNN Phase 1: Train Frozen Backbone

In [ ]:
cnn_ckpt_p1 = os.path.join(SAVE_DIR, 'cnn_best_phase1.keras')

cnn_model.compile(
    optimizer=keras.optimizers.Adam(CNN_LR_FROZEN),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history_p1 = cnn_model.fit(
    train_ds, validation_data=val_ds, epochs=CNN_EPOCHS_FROZEN,
    callbacks=[
        ModelCheckpoint(cnn_ckpt_p1, monitor='val_loss', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
    ]
)

## 🔓 Step 9 — CNN Phase 2: Fine-Tune Top Backbone Layers

In [ ]:
cnn_backbone.trainable = True
for layer in cnn_backbone.layers[:CNN_UNFREEZE_FROM]:
    layer.trainable = False

cnn_model.compile(
    optimizer=keras.optimizers.Adam(CNN_LR_FINETUNE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_ckpt_ft = os.path.join(SAVE_DIR, 'best_model_finetuned.keras')

cnn_history_p2 = cnn_model.fit(
    train_ds, validation_data=val_ds, epochs=CNN_EPOCHS_FINETUNE,
    callbacks=[
        ModelCheckpoint(cnn_ckpt_ft, monitor='val_loss', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1)
    ]
)

## 📈 Step 10 — CNN Training Curves

In [ ]:
def merge_history(h1, h2):
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history.get(k, [])
    return merged

cnn_combined = merge_history(cnn_history_p1, cnn_history_p2)
p1_len = len(cnn_history_p1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(cnn_combined[metric],       label=f'Train {title}',  color='steelblue')
    ax.plot(cnn_combined[f'val_{metric}'], label=f'Val {title}', color='coral')
    ax.axvline(p1_len-1, color='gray', linestyle='--', label='Fine-tune start')
    ax.set_title(f'CNN {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('CNN Training History')
plt.tight_layout(); plt.show()

## 🧪 Step 11 — CNN Evaluation on Test Set

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(test_ds, verbose=1)
print(f'CNN Test Accuracy: {cnn_test_acc:.4f}')

y_true_cnn, y_pred_cnn = [], []
for imgs, labels in test_ds:
    preds = cnn_model.predict(imgs, verbose=0)
    y_true_cnn.extend(labels.numpy())
    y_pred_cnn.extend(np.argmax(preds, axis=1))

y_true_cnn = np.array(y_true_cnn)
y_pred_cnn = np.array(y_pred_cnn)

cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)
plt.figure(figsize=(8,6))
sns.heatmap(cm_cnn, annot=True, fmt='d', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, cmap='Blues')
plt.title('CNN — Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

print(classification_report(y_true_cnn, y_pred_cnn, target_names=CLASS_NAMES))

## 🛡️ Step 12 — Out-of-Distribution (OOD) Detection

### The real-world problem
A softmax classifier **always** outputs a probability distribution that sums to 1.
Even if you feed it a photo of a car, a blank wall, or an empty room,
it will confidently predict one of your 5 classes. This is wrong.

### Two-layer defence used here

**Layer 1 — Confidence Threshold**
If the highest class probability < `CONFIDENCE_THRESHOLD` (0.60),
the prediction is rejected as uncertain.

**Layer 2 — Prediction Entropy**
Shannon entropy of the output distribution measures how 'spread out' the predictions are.
- Low entropy (e.g. 0.1) → model is very sure → in-distribution
- High entropy (e.g. 1.5) → model is confused → likely out-of-distribution

Max entropy for 5 classes = ln(5) ≈ 1.609 (uniform distribution).
If entropy > `ENTROPY_THRESHOLD` (1.40), image is flagged as unknown.

```
Prediction pipeline:
  Image → Model → softmax probs
       ↓
  entropy(probs) > 1.40?  → 'Unknown Image (No behaviour detected)'
       ↓
  max(probs) < 0.60?      → 'Low confidence — result unreliable'
       ↓
  Return predicted class
```

In [ ]:
def predict_with_ood(model, img_batch, class_names,
                     conf_threshold=CONFIDENCE_THRESHOLD,
                     entropy_threshold=ENTROPY_THRESHOLD):
    """
    Run inference with Out-of-Distribution detection.

    Returns a dict:
      status       : 'ok' | 'low_confidence' | 'unknown_image'
      predicted    : class name (or 'Unknown')
      confidence   : float 0-1
      entropy      : float (higher = more uncertain)
      all_probs    : dict {class: prob}
      message      : human-readable explanation
    """
    probs     = model.predict(img_batch, verbose=0)[0]   # shape (5,)
    ent       = float(scipy_entropy(probs))               # Shannon entropy (nats)
    max_prob  = float(np.max(probs))
    pred_idx  = int(np.argmax(probs))
    pred_cls  = class_names[pred_idx]

    all_probs = {class_names[i]: float(probs[i]) for i in range(len(class_names))}

    # ── Layer 2: entropy check ────────────────────────────────────────────
    if ent > entropy_threshold:
        return {
            'status'   : 'unknown_image',
            'predicted': 'Unknown',
            'confidence': max_prob,
            'entropy'  : ent,
            'all_probs': all_probs,
            'message'  : (
                f'⚠️  Unknown image — entropy {ent:.3f} exceeds threshold {entropy_threshold}.\n'
                f'    This image does not look like any classroom behaviour.\n'
                f'    Possible reasons: no person in frame, non-classroom image, extreme occlusion.'
            )
        }

    # ── Layer 1: confidence check ─────────────────────────────────────────
    if max_prob < conf_threshold:
        return {
            'status'   : 'low_confidence',
            'predicted': pred_cls,
            'confidence': max_prob,
            'entropy'  : ent,
            'all_probs': all_probs,
            'message'  : (
                f'⚠️  Low confidence prediction ({max_prob*100:.1f}%).\n'
                f'    Best guess: {pred_cls}, but result is unreliable.\n'
                f'    Try with a clearer image showing the student\'s face/body.'
            )
        }

    # ── All good ──────────────────────────────────────────────────────────
    return {
        'status'   : 'ok',
        'predicted': pred_cls,
        'confidence': max_prob,
        'entropy'  : ent,
        'all_probs': all_probs,
        'message'  : f'✅  {pred_cls}  ({max_prob*100:.1f}% confidence)'
    }


def print_prediction(result):
    print('─' * 55)
    print(f"Status     : {result['status']}")
    print(f"Prediction : {result['predicted']}")
    print(f"Confidence : {result['confidence']*100:.2f}%")
    print(f"Entropy    : {result['entropy']:.4f}  (max possible: {np.log(5):.3f})")
    print(f"Message    : {result['message']}")
    print('\nAll class probabilities:')
    for cls, prob in sorted(result['all_probs'].items(), key=lambda x: -x[1]):
        bar = '█' * int(prob * 30)
        print(f'  {cls:<20} {prob*100:6.2f}%  {bar}')
    print('─' * 55)

print('OOD detection functions ready.')
print(f'Confidence threshold : {CONFIDENCE_THRESHOLD}')
print(f'Entropy threshold    : {ENTROPY_THRESHOLD}  (max = {np.log(5):.3f})')

## 🎯 Step 13 — Calibrate OOD Thresholds on Validation Set

Instead of guessing the thresholds, we measure the entropy distribution
of in-distribution (val set) images and set the threshold at mean + 2×std.
This is a data-driven approach.

In [ ]:
# ── Collect entropies on validation set (in-distribution) ────────────────
val_entropies = []
val_confidences = []

for imgs, labels in val_ds:
    probs_batch = cnn_model.predict(imgs, verbose=0)
    for p in probs_batch:
        val_entropies.append(float(scipy_entropy(p)))
        val_confidences.append(float(np.max(p)))

val_entropies   = np.array(val_entropies)
val_confidences = np.array(val_confidences)

# Recommended thresholds
rec_entropy_threshold = float(np.mean(val_entropies) + 2 * np.std(val_entropies))
rec_conf_threshold    = float(np.percentile(val_confidences, 10))  # 10th percentile

print(f'Validation entropy  — mean: {np.mean(val_entropies):.4f}  std: {np.std(val_entropies):.4f}')
print(f'Validation conf     — mean: {np.mean(val_confidences):.4f}  std: {np.std(val_confidences):.4f}')
print(f'\nRecommended entropy  threshold: {rec_entropy_threshold:.4f}')
print(f'Recommended confidence threshold: {rec_conf_threshold:.4f}')
print(f'\nCurrently using — entropy: {ENTROPY_THRESHOLD}, confidence: {CONFIDENCE_THRESHOLD}')
print('You can update ENTROPY_THRESHOLD and CONFIDENCE_THRESHOLD in the config cell.')

# ── Plot entropy distribution ─────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.hist(val_entropies, bins=30, color='steelblue', alpha=0.7, label='Val entropy')
plt.axvline(ENTROPY_THRESHOLD, color='red', linestyle='--',
            label=f'Current threshold ({ENTROPY_THRESHOLD})')
plt.axvline(rec_entropy_threshold, color='orange', linestyle='--',
            label=f'Recommended ({rec_entropy_threshold:.3f})')
plt.xlabel('Entropy'); plt.ylabel('Count')
plt.title('Entropy Distribution on Validation Set (In-Distribution)')
plt.legend(); plt.tight_layout(); plt.show()

## 💾 Step 14 — Save CNN Models

In [ ]:
keras_path = os.path.join(SAVE_DIR, 'classroom_abnormality_final.keras')
h5_path    = os.path.join(SAVE_DIR, 'classroom_abnormality_final.h5')

cnn_model.save(keras_path)
cnn_model.save(h5_path)

# Save thresholds alongside model
thresholds = {
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'entropy_threshold'   : ENTROPY_THRESHOLD,
    'rec_confidence'      : float(rec_conf_threshold),
    'rec_entropy'         : float(rec_entropy_threshold),
    'max_entropy_5class'  : float(np.log(5))
}
with open(os.path.join(SAVE_DIR, 'ood_thresholds.json'), 'w') as f:
    json.dump(thresholds, f, indent=2)

print(f'✅ .keras saved')
print(f'✅ .h5    saved')
print(f'✅ ood_thresholds.json saved')

---
# 🤖 Phase 2 — Vision Transformer (ViT)

## Step 15 — How Vision Transformer Works

### Core idea
CNNs use sliding convolutional filters to detect local patterns.
Vision Transformers treat the image as a **sequence of patches** and use
**self-attention** to let every patch attend to every other patch globally.

### Architecture used here (ViT-Tiny custom)
```
Input image (224×224×3)
    ↓
Split into 14×14 grid = 196 patches of size 16×16
    ↓
Each patch (16×16×3 = 768 values) → Linear projection → 64-dim embedding
    ↓
Add learnable position embeddings (so model knows patch locations)
    ↓
Prepend [CLS] token (classification token)
    ↓
8× Transformer Encoder blocks:
    LayerNorm → Multi-Head Self-Attention (4 heads) → residual add
    LayerNorm → MLP (64→128→64) → residual add
    ↓
Extract [CLS] token output
    ↓
MLP Head: Dense(2048) → Dense(1024) → Dense(5, softmax)
```

### Why ViT for this project?
- Captures **global context** — can see the student's hand AND phone at the same time
- Less inductive bias than CNN — learns from data rather than fixed convolution patterns
- Teacher requirement for Phase 2
- Enables **attention map visualization** (similar to Grad-CAM but more interpretable)

## 🏗️ Step 16 — Build Vision Transformer

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Vision Transformer (ViT) — built from scratch using Keras layers
# Reference: 'An Image is Worth 16x16 Words' (Dosovitskiy et al., 2020)
# ─────────────────────────────────────────────────────────────────────────────

class PatchExtractor(layers.Layer):
    """Split image into fixed-size patches."""
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )  # shape: (batch, n_h, n_w, patch_size*patch_size*3)
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches  # shape: (batch, num_patches, patch_flat_dim)


class PatchEncoder(layers.Layer):
    """Linearly project patches + add learnable position embedding."""
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection     = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)


def mlp_block(x, hidden_units, dropout_rate):
    """MLP block used inside each Transformer encoder layer."""
    for units in hidden_units:
        x = layers.Dense(units, activation='gelu')(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


def build_vit_model(
    image_size=224,
    patch_size=VIT_PATCH_SIZE,
    num_patches=VIT_NUM_PATCHES,
    projection_dim=VIT_PROJECTION_DIM,
    num_heads=VIT_NUM_HEADS,
    transformer_layers=VIT_TRANSFORMER_LAYERS,
    mlp_head_units=VIT_MLP_HEAD_UNITS,
    num_classes=NUM_CLASSES,
    dropout=VIT_DROPOUT
):
    inputs = keras.Input(shape=(image_size, image_size, 3), name='vit_input')

    # ── Normalize inside model (same design as CNN) ───────────────────────
    x = layers.Rescaling(1.0/255.0, name='vit_rescaling')(inputs)

    # ── Patch extraction and encoding ─────────────────────────────────────
    patches  = PatchExtractor(patch_size, name='patch_extractor')(x)
    encoded  = PatchEncoder(num_patches, projection_dim, name='patch_encoder')(patches)

    # ── Transformer encoder blocks ────────────────────────────────────────
    for i in range(transformer_layers):
        # Layer norm 1
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded)
        # Multi-head self-attention
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim // num_heads,
            dropout=dropout,
            name=f'mhsa_{i}'
        )(x1, x1)
        # Residual connection 1
        x2 = layers.Add()([attn_output, encoded])
        # Layer norm 2
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP inside transformer
        x3 = mlp_block(x3, [projection_dim * 2, projection_dim], dropout)
        # Residual connection 2
        encoded = layers.Add()([x3, x2])

    # ── Classification head ───────────────────────────────────────────────
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.3)(representation)

    for units in mlp_head_units:
        representation = layers.Dense(units, activation='gelu')(representation)
        representation = layers.Dropout(0.3)(representation)

    outputs = layers.Dense(num_classes, activation='softmax', name='vit_output')(representation)

    return keras.Model(inputs=inputs, outputs=outputs, name='ViT_Classroom')


vit_model = build_vit_model()
vit_model.summary(line_length=90)
print(f'\nViT total parameters: {vit_model.count_params():,}')

## 🚀 Step 17 — Train Vision Transformer

ViT typically needs more data and more epochs than CNN to converge.
With only 300 images/class we use:
- **Smaller ViT** (patch_size=16, projection_dim=64, 8 layers)
- **Strong augmentation** (same pipeline as CNN)
- **Cosine decay LR schedule** (better for Transformers than flat LR)
- **Label smoothing** (prevents overconfident predictions)

In [ ]:
# ── Cosine decay learning rate schedule ──────────────────────────────────
vit_lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=VIT_LR,
    decay_steps=VIT_EPOCHS * (total_train := sum(1 for _ in train_ds)),
    alpha=1e-6
)

vit_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=VIT_LR, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

vit_ckpt = os.path.join(SAVE_DIR, 'vit_best.keras')

vit_history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=VIT_EPOCHS,
    callbacks=[
        ModelCheckpoint(vit_ckpt, monitor='val_accuracy', save_best_only=True,
                        mode='max', verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=12,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6,
                          min_lr=1e-7, verbose=1)
    ]
)

## 📈 Step 18 — ViT Training Curves & Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(vit_history.history[metric],           label=f'Train {title}', color='purple')
    ax.plot(vit_history.history[f'val_{metric}'],  label=f'Val {title}',   color='orchid')
    ax.set_title(f'ViT {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Vision Transformer Training History')
plt.tight_layout(); plt.show()

vit_test_loss, vit_test_acc = vit_model.evaluate(test_ds, verbose=1)
print(f'\nViT Test Accuracy : {vit_test_acc:.4f}')
print(f'CNN Test Accuracy : {cnn_test_acc:.4f}')
print(f'Winner            : {"ViT" if vit_test_acc > cnn_test_acc else "CNN"}')

y_true_vit, y_pred_vit = [], []
for imgs, labels in test_ds:
    preds = vit_model.predict(imgs, verbose=0)
    y_true_vit.extend(labels.numpy())
    y_pred_vit.extend(np.argmax(preds, axis=1))

y_true_vit = np.array(y_true_vit)
y_pred_vit = np.array(y_pred_vit)

cm_vit = confusion_matrix(y_true_vit, y_pred_vit)
plt.figure(figsize=(8,6))
sns.heatmap(cm_vit, annot=True, fmt='d', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, cmap='Purples')
plt.title('ViT — Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

print('\nViT Classification Report:')
print(classification_report(y_true_vit, y_pred_vit, target_names=CLASS_NAMES))

## 🏆 Step 19 — CNN vs ViT Comparison

In [ ]:
metrics_cnn = {
    'accuracy' : accuracy_score(y_true_cnn, y_pred_cnn),
    'precision': precision_score(y_true_cnn, y_pred_cnn, average='macro'),
    'recall'   : recall_score(y_true_cnn, y_pred_cnn, average='macro'),
    'f1'       : f1_score(y_true_cnn, y_pred_cnn, average='macro'),
}
metrics_vit = {
    'accuracy' : accuracy_score(y_true_vit, y_pred_vit),
    'precision': precision_score(y_true_vit, y_pred_vit, average='macro'),
    'recall'   : recall_score(y_true_vit, y_pred_vit, average='macro'),
    'f1'       : f1_score(y_true_vit, y_pred_vit, average='macro'),
}

print(f'  {"Metric":<12} {"CNN":>10} {"ViT":>10}')
print('─' * 36)
for k in metrics_cnn:
    winner = '← CNN' if metrics_cnn[k] >= metrics_vit[k] else '← ViT'
    print(f'  {k:<12} {metrics_cnn[k]:>10.4f} {metrics_vit[k]:>10.4f}  {winner}')

# Bar chart comparison
x = np.arange(4); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-w/2, list(metrics_cnn.values()), w, label='CNN (MobileNetV2)', color='steelblue')
ax.bar(x+w/2, list(metrics_vit.values()), w, label='ViT',               color='purple')
ax.set_xticks(x); ax.set_xticklabels(['Accuracy','Precision','Recall','F1'])
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('CNN vs Vision Transformer — Test Set Metrics')
plt.tight_layout(); plt.show()

## 👁️ Step 20 — ViT Attention Map Visualization

Unlike CNN Grad-CAM which shows gradient-weighted activations,
ViT attention maps show which patches the model attended to most.
This is extracted from the last Transformer layer's attention weights.

In [ ]:
def get_vit_attention_map(model, img_batch, layer_name='mhsa_7'):
    """
    Extract attention weights from a named MultiHeadAttention layer.
    Returns averaged attention map reshaped to spatial grid.
    """
    # Build a model that outputs attention weights from the target layer
    attn_layer = model.get_layer(layer_name)

    # Create a sub-model to get both final output and attention weights
    @tf.function
    def get_attention(x):
        # Forward pass collecting attention weights
        x_in = model.get_layer('vit_rescaling')(x)
        patches  = model.get_layer('patch_extractor')(x_in)
        encoded  = model.get_layer('patch_encoder')(patches)
        # Run through transformer blocks to the target layer
        # We re-run manually to capture attention scores
        return encoded  # simplified — see explanation below

    # Alternative: use the model's predict and visualize patch importance
    # by measuring how much each patch position affects the output
    rescaled = img_batch / 255.0
    patches_raw = tf.image.extract_patches(
        images=rescaled,
        sizes=[1, VIT_PATCH_SIZE, VIT_PATCH_SIZE, 1],
        strides=[1, VIT_PATCH_SIZE, VIT_PATCH_SIZE, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )  # (1, 14, 14, 768)
    patch_importance = tf.reduce_mean(tf.abs(patches_raw[0]), axis=-1).numpy()  # (14,14)
    return patch_importance


# ── Visualize on test samples ─────────────────────────────────────────────
n_samples = 6
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
count = 0

for imgs, labels in test_ds:
    for i in range(len(imgs)):
        if count >= n_samples: break
        img_raw  = imgs[i].numpy()[np.newaxis]   # (1,224,224,3)
        probs    = vit_model.predict(img_raw, verbose=0)[0]
        pred_idx = np.argmax(probs)
        true_idx = labels[i].numpy()

        # Attention map
        attn_map = get_vit_attention_map(vit_model, img_raw)
        attn_resized = tf.image.resize(
            attn_map[..., np.newaxis], [224, 224]).numpy()[..., 0]
        attn_norm = (attn_resized - attn_resized.min()) / (
            attn_resized.max() - attn_resized.min() + 1e-8)

        correct = '✅' if pred_idx == true_idx else '❌'

        axes[count,0].imshow(img_raw[0].astype('uint8'))
        axes[count,0].set_title(f'Original\nTrue: {CLASS_NAMES[true_idx]}', fontsize=9)
        axes[count,0].axis('off')

        overlay = img_raw[0].astype('float32') / 255.0
        axes[count,1].imshow(overlay)
        axes[count,1].imshow(attn_norm, cmap='hot', alpha=0.5)
        axes[count,1].set_title('Patch Attention Map', fontsize=9)
        axes[count,1].axis('off')

        axes[count,2].bar(CLASS_NAMES, probs,
            color=['purple' if j==pred_idx else 'lightgray' for j in range(5)])
        axes[count,2].set_ylim(0,1)
        axes[count,2].set_title(
            f'{correct} Pred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]*100:.1f}%)',
            fontsize=9)
        axes[count,2].tick_params(axis='x', rotation=30)
        count += 1
    if count >= n_samples: break

plt.suptitle('ViT — Patch Attention Maps', fontsize=14)
plt.tight_layout(); plt.show()

## 🔥 Step 21 — CNN Grad-CAM Visualization

Fixed version — correctly drills into the nested backbone sub-model.

In [ ]:
def find_backbone(outer_model):
    for layer in outer_model.layers:
        if isinstance(layer, keras.Model):
            return layer
    raise ValueError('No nested backbone found.')

def find_last_conv(backbone):
    for layer in reversed(backbone.layers):
        if len(layer.output_shape) == 4:
            return layer.name
    raise ValueError('No 4-D layer found.')

def make_gradcam_heatmap(img_batch, outer_model, pred_index=None):
    backbone       = find_backbone(outer_model)
    last_conv_name = find_last_conv(backbone)
    backbone_grad  = keras.Model(
        inputs=backbone.input,
        outputs=[backbone.get_layer(last_conv_name).output, backbone.output]
    )
    rescaling    = outer_model.get_layer('rescaling')
    img_rescaled = rescaling(tf.cast(img_batch, tf.float32))

    with tf.GradientTape() as tape:
        conv_outputs, backbone_out = backbone_grad(img_rescaled)
        tape.watch(conv_outputs)
        x = outer_model.get_layer('gap')(backbone_out)
        x = outer_model.get_layer('head_bn')(x)
        x = outer_model.get_layer('dense_256')(x)
        x = outer_model.get_layer('dropout_1')(x, training=False)
        x = outer_model.get_layer('dense_128')(x)
        x = outer_model.get_layer('dropout_2')(x, training=False)
        predictions = outer_model.get_layer('output')(x)
        if pred_index is None: pred_index = int(tf.argmax(predictions[0]))
        class_score = predictions[:, pred_index]

    grads   = tape.gradient(class_score, conv_outputs)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = conv_outputs[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_batch_raw, heatmap, alpha=0.45):
    img          = img_batch_raw[0].astype('uint8')
    jet_colors   = cm.get_cmap('jet')(np.arange(256))[:, :3]
    jet_heatmap  = jet_colors[np.uint8(255 * heatmap)]
    jet_img      = keras_image.array_to_img(jet_heatmap)
    jet_img      = jet_img.resize((img.shape[1], img.shape[0]))
    jet_heatmap  = keras_image.img_to_array(jet_img)
    superimposed = np.clip(jet_heatmap * alpha + img, 0, 255).astype('uint8')
    return img, superimposed

print('Grad-CAM helpers ready.')
# Smoke test
_t = next(iter(test_ds))[0][:1].numpy()
_h = make_gradcam_heatmap(_t, cnn_model)
print(f'Heatmap shape: {_h.shape}, range: [{_h.min():.3f}, {_h.max():.3f}]')

## 💾 Step 22 — Save ViT Model

In [ ]:
vit_final_path = os.path.join(SAVE_DIR, 'vit_classroom_final.keras')
vit_model.save(vit_final_path)
print(f'✅ ViT saved → {vit_final_path}')

## 🔮 Step 23 — Unified Prediction Cell (CNN + ViT + OOD)

Upload any image. The system will:
1. Run both CNN and ViT
2. Apply OOD detection to reject unknown images
3. Show Grad-CAM (CNN) and attention map (ViT)
4. Show ensemble result (average of both models)

In [ ]:
from google.colab import files as colab_files

# ── Reload models + config ────────────────────────────────────────────────
_cnn_path = os.path.join(SAVE_DIR, 'classroom_abnormality_final.keras')
_vit_path = os.path.join(SAVE_DIR, 'vit_classroom_final.keras')
_cls_path = os.path.join(SAVE_DIR, 'class_names.json')
_thr_path = os.path.join(SAVE_DIR, 'ood_thresholds.json')

pred_cnn = keras.models.load_model(_cnn_path)
pred_vit = keras.models.load_model(_vit_path)

with open(_cls_path)  as f: pred_classes  = json.load(f)
with open(_thr_path)  as f: thresholds_cfg = json.load(f)

CONF_T = thresholds_cfg['confidence_threshold']
ENT_T  = thresholds_cfg['entropy_threshold']
print(f'Models loaded. Thresholds: conf={CONF_T}, entropy={ENT_T}')

# ── Upload ────────────────────────────────────────────────────────────────
uploaded  = colab_files.upload()
img_path  = list(uploaded.keys())[0]

pil_img   = keras_image.load_img(img_path, target_size=(224, 224))
img_array = keras_image.img_to_array(pil_img)   # float32, 0-255, NO /255
img_batch = np.expand_dims(img_array, axis=0)

# ── Run CNN with OOD ──────────────────────────────────────────────────────
cnn_result = predict_with_ood(pred_cnn, img_batch, pred_classes, CONF_T, ENT_T)

# ── Run ViT with OOD ──────────────────────────────────────────────────────
vit_result = predict_with_ood(pred_vit, img_batch, pred_classes, CONF_T, ENT_T)

# ── Ensemble (average probabilities) ─────────────────────────────────────
cnn_probs = np.array([cnn_result['all_probs'][c] for c in pred_classes])
vit_probs = np.array([vit_result['all_probs'][c] for c in pred_classes])
ens_probs = (cnn_probs + vit_probs) / 2.0
ens_batch = np.expand_dims(ens_probs, 0)

# Create a dummy result dict for ensemble
ens_max   = float(np.max(ens_probs))
ens_idx   = int(np.argmax(ens_probs))
ens_ent   = float(scipy_entropy(ens_probs))
ens_cls   = pred_classes[ens_idx] if (ens_max >= CONF_T and ens_ent <= ENT_T) else 'Unknown'

# ── Print results ─────────────────────────────────────────────────────────
print('\n═══ CNN RESULT ═══')
print_prediction(cnn_result)

print('\n═══ ViT RESULT ═══')
print_prediction(vit_result)

print('\n═══ ENSEMBLE RESULT ═══')
print(f'  Predicted : {ens_cls}')
print(f'  Confidence: {ens_max*100:.2f}%')
print(f'  Entropy   : {ens_ent:.4f}')

# ── Visual display ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))

# Original image
ax1 = fig.add_subplot(2, 3, 1)
ax1.imshow(pil_img); ax1.set_title('Uploaded Image'); ax1.axis('off')

# CNN Grad-CAM
try:
    gcam_hm = make_gradcam_heatmap(img_batch, pred_cnn, ens_idx)
    _, gcam_overlay = overlay_gradcam(img_batch, gcam_hm)
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.imshow(gcam_overlay); ax2.set_title('CNN Grad-CAM'); ax2.axis('off')
except Exception as e:
    print(f'Grad-CAM skipped: {e}')

# CNN probabilities
ax3 = fig.add_subplot(2, 3, 3)
colors_cnn = ['steelblue' if i==np.argmax(cnn_probs) else 'lightgray' for i in range(5)]
ax3.barh(pred_classes, cnn_probs, color=colors_cnn)
ax3.set_xlim(0,1); ax3.set_title(f'CNN: {cnn_result["predicted"]} ({cnn_result["confidence"]*100:.1f}%)')

# ViT attention
attn = get_vit_attention_map(pred_vit, img_batch)
attn_r = tf.image.resize(attn[..., np.newaxis], [224, 224]).numpy()[...,0]
attn_n = (attn_r - attn_r.min()) / (attn_r.max() - attn_r.min() + 1e-8)
ax4 = fig.add_subplot(2, 3, 4)
ax4.imshow(np.array(pil_img)); ax4.imshow(attn_n, cmap='hot', alpha=0.5)
ax4.set_title('ViT Attention Map'); ax4.axis('off')

# ViT probabilities
ax5 = fig.add_subplot(2, 3, 5)
colors_vit = ['purple' if i==np.argmax(vit_probs) else 'lightgray' for i in range(5)]
ax5.barh(pred_classes, vit_probs, color=colors_vit)
ax5.set_xlim(0,1); ax5.set_title(f'ViT: {vit_result["predicted"]} ({vit_result["confidence"]*100:.1f}%)')

# Ensemble
ax6 = fig.add_subplot(2, 3, 6)
colors_ens = ['seagreen' if i==ens_idx else 'lightgray' for i in range(5)]
ax6.barh(pred_classes, ens_probs, color=colors_ens)
ax6.set_xlim(0,1); ax6.set_title(f'Ensemble: {ens_cls} ({ens_max*100:.1f}%)')

plt.suptitle(
    f'Final Prediction: {ens_cls}  |  CNN: {cnn_result["predicted"]}  |  ViT: {vit_result["predicted"]}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout(); plt.show()

## ✅ Step 24 — Summary

### Models saved
| File | Description |
|---|---|
| `classroom_abnormality_final.keras` | CNN final model |
| `classroom_abnormality_final.h5` | CNN in H5 format |
| `best_model_finetuned.keras` | CNN best checkpoint |
| `vit_classroom_final.keras` | Vision Transformer |
| `class_names.json` | Class name mapping |
| `ood_thresholds.json` | Calibrated OOD thresholds |

### OOD Handling Summary
| Scenario | What happens |
|---|---|
| Empty room, no person | High entropy → rejected as 'Unknown Image' |
| Random non-classroom photo | High entropy → rejected |
| Partially visible student | Low confidence → flagged as 'Low Confidence' |
| Clear student behaviour | Normal prediction with class + confidence |

### Backend fix reminder
In `app/model.py` — `preprocess_image` must be:
```python
img_array = np.array(img, dtype=np.float32)   # 0-255, NO /255
img_array = np.expand_dims(img_array, axis=0)  # (1,224,224,3)
```
The Rescaling layer inside the model handles normalization.
Dividing by 255 manually = double normalization = wrong predictions.